In [47]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [48]:
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
LANGCHAIN_API_KEY=os.getenv("LANGCHAIN_API_KEY")
LANGCHAIN_PROJECT=os.getenv("LANGCHAIN_PROJECT")

In [49]:
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["TAVILY_API_KEY"]= TAVILY_API_KEY
os.environ["GROQ_API_KEY"]= GROQ_API_KEY
os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"]=LANGCHAIN_PROJECT

In [18]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2934.77it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [50]:
from langchain_groq import ChatGroq
llm_groq=ChatGroq(model="qwen/qwen3-32b")

# Predefine tools

In [51]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wikipedia = WikipediaAPIWrapper()
wikipedia_tool = WikipediaQueryRun(api_wrapper=wikipedia)

In [52]:
print(wikipedia_tool.run("Artificial Intelligence"))

Page: Artificial intelligence
Summary: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
High-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., language models and AI art); and superhuman play and analysis in strategy games (e.g., chess and Go). However, many AI applications are not perceived as such: "A lot of cutting-edge AI has filtered into genera

In [53]:
from langchain_community.tools.tavily_search import TavilySearchResults
tavily_tool=TavilySearchResults()

C:\Users\USER\AppData\Local\Temp\ipykernel_11376\1612002888.py:2: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool=TavilySearchResults()


# Creating Custom Tool to expirement with agent

In [54]:
from langchain.tools import tool
@tool
def get_word_length(word : str)->int:
    """Returns the length of a word"""
    return len(word)

In [55]:
get_word_length.invoke("LangGraph")

9

In [56]:
@tool
def multiply_numbers(num1: int, num2: int) -> int:
    """Multiplies two numbers and returns the result"""
    return num1 * num2

In [57]:
multiply_numbers.args

{'num1': {'title': 'Num1', 'type': 'integer'},
 'num2': {'title': 'Num2', 'type': 'integer'}}

In [58]:
multiply_numbers.invoke({"num1": 5, "num2": 7})

35

In [59]:
@tool
def summation_2_numbers(num1: int, num2: int) -> int:
    """Adds two numbers and returns the result"""
    return num1 + num2

In [60]:
summation_2_numbers.args

{'num1': {'title': 'Num1', 'type': 'integer'},
 'num2': {'title': 'Num2', 'type': 'integer'}}

In [61]:
summation_2_numbers.invoke({"num1": 10, "num2": 15})

25

In [62]:
tools=[tavily_tool, multiply_numbers, summation_2_numbers,get_word_length, wikipedia_tool]

In [63]:
prompt="You are a helpful assistant. Use tools when needed “If one tool is insufficient, try another relevant tool before final answer"

In [64]:
from langchain.agents import create_agent

In [65]:
agent = create_agent(
    model=llm_groq,
    tools=tools,
    system_prompt=prompt
)

In [66]:
result=agent.invoke({
    'messages':[
        {'role':'user','content':'What is the sum of 12 and 20?'}
    ]
})

In [67]:
result

{'messages': [HumanMessage(content='What is the sum of 12 and 20?', additional_kwargs={}, response_metadata={}, id='0477faa9-ecfc-49f6-953a-ad8415eb84ab'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for the sum of 12 and 20. Let me check which tool I can use here. The available functions include summation_2_numbers, which is exactly for adding two numbers. The parameters required are num1 and num2, both integers. So I should call that function with 12 and 20 as the arguments. I don't need to use any other tool here since it's a straightforward addition. Let me make sure the inputs are correct. Yep, 12 and 20 are integers. Alright, time to format the tool call.\n", 'tool_calls': [{'id': 'shsnjr6er', 'function': {'arguments': '{"num1":12,"num2":20}', 'name': 'summation_2_numbers'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 161, 'prompt_tokens': 527, 'total_tokens': 688, 'completion_time': 0.252526639, 'co

In [68]:
print(result['messages'][-1].content)

The sum of 12 and 20 is **32**.


In [24]:
result=agent.invoke({
    'messages':[
        {'role':'user','content':'What is the sum of 12,20 and 70?'}
    ]
})

In [25]:
result

{'messages': [HumanMessage(content='What is the sum of 12,20 and 70?', additional_kwargs={}, response_metadata={}, id='4ec210a0-8092-49b6-9c46-9a053c4c15ad'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for the sum of 12, 20, and 70. Let me see. The tools provided include a summation function called summation_2_numbers, which adds two numbers. But here there are three numbers. Hmm, maybe I can use the function twice. First, add 12 and 20, which gives 32. Then add 32 and 70. Alternatively, check if there's another function that can handle more numbers, but looking at the tools, the summation function only takes two numbers. So I'll need to call it twice. Let me confirm the steps: 12 + 20 = 32, then 32 +70=102. The final answer should be 102. Since the functions are available, I should structure the tool calls accordingly.\n", 'tool_calls': [{'id': 'b5knvxt09', 'function': {'arguments': '{"num1":12,"num2":20}', 'name': 'summation_2_numbers'},

In [26]:
print(result['messages'][-1].content)

The sum of 12, 20, and 70 is **102**. 

<step-by-step>
1. First, add 12 and 20:  
   $12 + 20 = 32$  
2. Then, add the result to 70:  
   $32 + 70 = 102$  
</step-by-step>


In [27]:
# experiment to use all  available tools given for an agent
result=agent.invoke({
    'messages':[
        {'role':'user','content':""" Find the length of the word "Japan".

Multiply that length by 12.

Add 20 to the result.

Use a web search tool to find one recent fact about Japan.

Also get a short summary of Japan from Wikipedia.

Finally, combine everything into one final answer including:
- the word length
- the multiplication result
- the final sum
- the recent fact
- the Wikipedia summary"""}
    ]
})

d:\LangGraph-end-to-end\.venv\Lib\site-packages\wikipedia\wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("html.parser"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file d:\LangGraph-end-to-end\.venv\Lib\site-packages\wikipedia\wikipedia.py. To get rid of this warning, pass the additional argument 'features="html.parser"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


In [28]:
result

{'messages': [HumanMessage(content=' Find the length of the word "Japan".\n\nMultiply that length by 12.\n\nAdd 20 to the result.\n\nUse a web search tool to find one recent fact about Japan.\n\nAlso get a short summary of Japan from Wikipedia.\n\nFinally, combine everything into one final answer including:\n- the word length\n- the multiplication result\n- the final sum\n- the recent fact\n- the Wikipedia summary', additional_kwargs={}, response_metadata={}, id='a6ede5bd-9e61-42e1-aa9c-e041ee10a9e4'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s tackle this step by step. First, the user wants the length of the word "Japan". I\'ll use the get_word_length function for that. The word is 5 letters long.\n\nNext, multiply that length by 12. So 5 times 12. I\'ll use the multiply_numbers function here, which gives 60.\n\nThen add 20 to the result. Using summation_2_numbers with 60 and 20, that totals 80.\n\nNow, find a recent fact about Japan using a web sear

In [29]:
print(result['messages'][-1].content)

Here’s the combined final answer:

---

### **Word Length**:  
The word "Japan" has **5 letters**.

### **Math Calculations**:  
- **Multiplication**: $5 \times 12 = 60$  
- **Final Sum**: $60 + 20 = 80$  

### **Recent Fact (2023)**:  
In August 2023, Japan began releasing treated radioactive water from the Fukushima Daiichi Nuclear Plant into the ocean, as approved by the International Atomic Energy Agency (IAEA). This move faced criticism from local fishermen, China, and Russia, with China banning Japanese seafood imports.  

### **Wikipedia Summary**:  
Japan is an island nation in East Asia, comprising four major islands and 14,121 smaller islands. With a population of ~123 million, it is the 11th most populous country. Historically, it transitioned from feudal shogunate rule to a modern state after the Meiji Restoration. Post-WWII, Japan became a G7 member and a global leader in technology and manufacturing. Today, it faces challenges like population decline and economic stagnati

Here’s the combined final answer:

---

### **Word Length**:  
The word "Japan" has **5 letters**.

### **Math Calculations**:  
- **Multiplication**: $5 \times 12 = 60$  
- **Final Sum**: $60 + 20 = 80$  

### **Recent Fact (2023)**:  
In August 2023, Japan began releasing treated radioactive water from the Fukushima Daiichi Nuclear Plant into the ocean, as approved by the International Atomic Energy Agency (IAEA). This move faced criticism from local fishermen, China, and Russia, with China banning Japanese seafood imports.  

### **Wikipedia Summary**:  
Japan is an island nation in East Asia, comprising four major islands and 14,121 smaller islands. With a population of ~123 million, it is the 11th most populous country. Historically, it transitioned from feudal shogunate rule to a modern state after the Meiji Restoration. Post-WWII, Japan became a G7 member and a global leader in technology and manufacturing. Today, it faces challenges like population decline and economic stagnation but remains a cultural powerhouse, renowned for its contributions to art, cuisine, and pop culture (e.g., anime, robotics).  

--- 

Let me know if you need further details! 🌸

Connecting RAG  As a Tool

In [69]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [70]:

loader = WebBaseLoader("https://blog.langchain.dev/langgraph-studio-the-first-agent-ide/")
docs=loader.load()

In [71]:
docs

[Document(metadata={'source': 'https://blog.langchain.dev/langgraph-studio-the-first-agent-ide/', 'title': 'LangGraph Studio: The first agent IDE', 'description': 'LangGraph Studio provides a specialized agent IDE for visualizing, interacting with, and debugging complex agentic applications. See how to use it on your desktop today.', 'language': 'en'}, page_content="\n\n\nLangGraph Studio: The first agent IDE\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSkip to content\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nWebsite\n\n\n\n\nDocs\n\n\n\n\nCase Studies\n\n\n\n\nHarrison's In the Loop Series\n\n\n\n\nTry LangSmith\n\n\n\n\n\nSign in\nSubscribe\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLangGraph Studio: The first agent IDE\nLangGraph Studio provides a specialized agent IDE for visualizing, interacting with, and debugging complex agentic applications. See how to use it on your desktop today.\n\n4 min read\nAug 1, 2024\n\n\n\n\n\nLLMs have

In [72]:
documents=RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=100
).split_documents(docs)

In [73]:
len(documents)

17

In [74]:
vectorstore=FAISS.from_documents(documents, embeddings)

In [75]:
retriever=vectorstore.as_retriever()

In [76]:
retriever.invoke("What is LangGraph Studio?")

[Document(id='d691b811-c502-4c0f-9f6f-9fe69fe6ca84', metadata={'source': 'https://blog.langchain.dev/langgraph-studio-the-first-agent-ide/', 'title': 'LangGraph Studio: The first agent IDE', 'description': 'LangGraph Studio provides a specialized agent IDE for visualizing, interacting with, and debugging complex agentic applications. See how to use it on your desktop today.', 'language': 'en'}, page_content="can\xa0sign up for LangSmith\xa0today to try out LangGraph Studio for free.We'd also love your feedback - drop us a line at hello@langchain.dev or on Twitter to share your thoughts."),
 Document(id='b295518c-9e07-46eb-9960-51a91ceac06b', metadata={'source': 'https://blog.langchain.dev/langgraph-studio-the-first-agent-ide/', 'title': 'LangGraph Studio: The first agent IDE', 'description': 'LangGraph Studio provides a specialized agent IDE for visualizing, interacting with, and debugging complex agentic applications. See how to use it on your desktop today.', 'language': 'en'}, page_

In [86]:
from langchain.tools import tool

@tool
def retrieve_blog_posts_LangGraph_Studio(query: str) -> str:
    """Search and return information about langraph_studio_search,For any questions about LangGraph Studio, use this tool to retrieve relevant information from the LangGraph Studio blog post."""
    docs = retriever.invoke(query)  # Your retriever here
    return "\n\n".join([doc.page_content for doc in docs])

retriever_tool_LangGraph = retrieve_blog_posts_LangGraph_Studio.invoke('What is LangGraph Studio?')

In [87]:
print(retriever_tool_LangGraph)

can sign up for LangSmith today to try out LangGraph Studio for free.We'd also love your feedback - drop us a line at hello@langchain.dev or on Twitter to share your thoughts.

is coming soon.After you download and open LangGraph Studio, you will be prompted to log in with your LangSmith account. All users of LangSmith (including those with free accounts) currently have access to LangGraph Studio while it is in beta. You can sign up for a LangSmith account here.After downloading LangSmith, you can open a directory. At a bare minimum, this directory needs to contain a Python file with a graph defined in it. Next, you will need to create a langgraph.json file containing details

an iterative process, by letting you interact with and manipulate the state at that point in time.While there is much more to explore, we're excited to introduce LangGraph Studio to start with bringing some of the core features of an agent IDE to the world.How to use LangGraph StudioLangGraph Studio is a desktop 

In [ ]:
tools=[retrieve_blog_posts_LangGraph_Studio, tavily_tool, multiply_numbers, summation_2_numbers,get_word_length, wikipedia_tool]

In [89]:
tools

[StructuredTool(name='retrieve_blog_posts_LangGraph_Studio', description='Search and return information about langraph_studio_search,For any questions about LangGraph Studio, use this tool to retrieve relevant information from the LangGraph Studio blog post.', args_schema=<class 'langchain_core.utils.pydantic.retrieve_blog_posts_LangGraph_Studio'>, func=<function retrieve_blog_posts_LangGraph_Studio at 0x000002095B2C9B20>),
 TavilySearchResults(api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'))),
 StructuredTool(name='multiply_numbers', description='Multiplies two numbers and returns the result', args_schema=<class 'langchain_core.utils.pydantic.multiply_numbers'>, func=<function multiply_numbers at 0x0000020959C7CF40>),
 StructuredTool(name='summation_2_numbers', description='Adds two numbers and returns the result', args_schema=<class 'langchain_core.utils.pydantic.summation_2_numbers'>, func=<function summation_2_numbers at 0x0000020959EDE660>),
 StructuredTo

In [98]:
agent = create_agent(
    model=llm_groq,
    tools=tools,
    system_prompt=prompt
)


In [99]:
result=agent.invoke({
    'messages':[{"role": "user", "content": "What is LangGraph Studio?"}]
})

In [100]:
result

{'messages': [HumanMessage(content='What is LangGraph Studio?', additional_kwargs={}, response_metadata={}, id='95f79db6-786c-4459-bc78-44ff5c9feccf'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking, "What is LangGraph Studio?" I need to figure out which tool to use here. Let me look at the available functions.\n\nThe first tool is retrieve_blog_posts_LangGraph_Studio, which is for searching information about LangGraph Studio from a blog post. The description says to use it for any questions about LangGraph Studio. Since the user is asking what it is, this seems like the right tool. The other tools are for math operations and word length, which don\'t apply here.\n\nSo I should call retrieve_blog_posts_LangGraph_Studio with the query parameter set to "What is LangGraph Studio?". That should fetch the relevant information from the blog post to answer the user\'s question.\n', 'tool_calls': [{'id': 'hf93fj0ra', 'function': {'arguments': '{"query

In [101]:
print(result['messages'][-1].content)

LangGraph Studio is a **desktop application** designed to simplify the development of agentic applications using LangGraph, a framework for building LLM-powered applications. Here's a concise overview:

### Key Features:
1. **Visual Debugging & Iteration**:  
   - Visualize and interact with agent graphs, enabling quick iterations by manipulating the state at any point in the process.
2. **Integration**:  
   - Works seamlessly with **LangSmith** (for authentication and tooling) and supports both **Python** and **JavaScript**.
   - Fully open-source, with compatibility for LangChain workflows.

### How to Access:
- **Currently in Beta**: Free to use with a LangSmith account (including free-tier users).  
- **Platform**: Available for **Apple Silicon** (Mac) via download. Support for additional platforms is upcoming.  
- **Setup**: Requires a Python file defining a graph and a `langgraph.json` configuration file.

### Future Plans:
- Expanding platform support.  
- Enhancing core agent 